# 基金每日净值与压力位监控

用于Kaggle定时执行的基金监控脚本，每日获取基金净值和压力位数据，通过飞书卡片推送。

## 使用说明
1. 在Kaggle上创建一个新的Notebook
2. import该文件
3. 填写飞书Webhook地址和需要监控的基金代码
4. 设置定时调度（建议每个交易日晚上8点后执行）

In [ ]:
# ============================================
# 配置区域 - 请在此处填写您的配置
# ============================================

# 飞书Webhook地址 - 请在此处填写您的飞书机器人Webhook地址
# 获取方式：在飞书群聊中添加自定义机器人，复制Webhook地址
FEISHU_WEBHOOK_URL = ""  # 示例: "https://open.feishu.cn/open-apis/bot/v2/hook/xxxxx"

# 需要监控的基金代码列表（支持多个基金）
FUND_CODES = [
    "012805",  # 示例：广发恒生科技ETF联接C
]

# 是否发送飞书通知（设置为False可仅测试数据获取，不发送通知）
ENABLE_NOTIFICATION = True

In [ ]:
# 安装必要的依赖
%pip install requests numpy -q

In [ ]:
import requests
import json
import re
import numpy as np
from datetime import datetime, timedelta
from typing import List, Dict, Optional, Tuple
import time

In [ ]:
class FundDataFetcher:
    """基金数据获取器"""
    
    def __init__(self, fund_code: str):
        self.fund_code = fund_code
        self.data = {}
    
    def fetch(self) -> Dict:
        """从天天基金网获取基金数据"""
        timestamp = datetime.now().strftime('%Y%m%d%H%M%S')
        url = f"https://fund.eastmoney.com/pingzhongdata/{self.fund_code}.js?v={timestamp}"
        
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
            'Referer': f'https://fund.eastmoney.com/{self.fund_code}.html'
        }
        
        try:
            response = requests.get(url, headers=headers, timeout=30)
            response.encoding = 'utf-8'
            
            if response.status_code == 200:
                self.js_content = response.text
                self._parse_data()
                return self.data
            else:
                print(f"获取基金 {self.fund_code} 数据失败: HTTP {response.status_code}")
                return {}
        except Exception as e:
            print(f"获取基金 {self.fund_code} 数据异常: {str(e)}")
            return {}
    
    def _parse_data(self):
        """解析JS内容提取关键数据"""
        # 基金基本信息
        self.data['name'] = self._extract_string('fS_name')
        self.data['code'] = self._extract_string('fS_code')
        
        # 收益率
        self.data['return_1m'] = self._extract_float('syl_1y')
        self.data['return_3m'] = self._extract_float('syl_3y')
        self.data['return_6m'] = self._extract_float('syl_6y')
        self.data['return_1y'] = self._extract_float('syl_1n')
        
        # 净值走势数据
        self.data['net_worth_trend'] = self._extract_json('Data_netWorthTrend')
        
    def _extract_string(self, var_name: str) -> str:
        """提取字符串变量"""
        pattern = rf'var {var_name}\s*=\s*"([^"]*)"'
        match = re.search(pattern, self.js_content)
        return match.group(1) if match else ''
    
    def _extract_float(self, var_name: str) -> float:
        """提取浮点数变量"""
        pattern = rf'var {var_name}\s*=\s*"([^"]*)"'
        match = re.search(pattern, self.js_content)
        if match:
            try:
                return float(match.group(1))
            except ValueError:
                return 0.0
        return 0.0
    
    def _extract_json(self, var_name: str) -> List[Dict]:
        """提取JSON数组"""
        pattern = rf'var {var_name}\s*=\s*(\[.*?\]);'
        match = re.search(pattern, self.js_content, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(1))
            except json.JSONDecodeError:
                return []
        return []

In [ ]:
class FundAnalyzer:
    """基金分析器 - 计算压力位等指标"""
    
    def __init__(self, data: Dict):
        self.data = data
        self.df = self._prepare_dataframe()
    
    def _prepare_dataframe(self) -> List[Dict]:
        """准备数据列表"""
        net_worth = self.data.get('net_worth_trend', [])
        records = []
        for item in net_worth:
            records.append({
                'date': datetime.fromtimestamp(item['x'] / 1000),
                'net_worth': item['y'],
                'daily_return': item.get('equityReturn', 0)
            })
        return records
    
    def get_latest_info(self) -> Dict:
        """获取最新净值信息"""
        if not self.df:
            return {}
        
        latest = self.df[-1]
        previous = self.df[-2] if len(self.df) > 1 else latest
        
        return {
            'date': latest['date'].strftime('%Y-%m-%d'),
            'net_worth': latest['net_worth'],
            'daily_return': latest['daily_return'],
            'previous_worth': previous['net_worth'],
            'change_amount': latest['net_worth'] - previous['net_worth']
        }
    
    def analyze_pressure_zones(self) -> Dict:
        """压力指标分析 - 识别支撑位和压力位，返回当前净值附近的区间"""
        if len(self.df) < 30:
            return {}
        
        values = [r['net_worth'] for r in self.df]
        max_val = max(values)
        min_val = min(values)
        current_price = self.df[-1]['net_worth']
        
        if max_val == min_val:
            return {}
        
        # 将净值区间划分为10个档位
        num_zones = 10
        zone_size = (max_val - min_val) / num_zones
        
        zones = []
        for i in range(num_zones):
            zone_min = min_val + i * zone_size
            zone_max = min_val + (i + 1) * zone_size
            zone_mid = (zone_min + zone_max) / 2
            
            # 找到该区间内的所有交易日
            zone_records = [
                (idx, r) for idx, r in enumerate(self.df)
                if zone_min <= r['net_worth'] < zone_max or (i == num_zones - 1 and r['net_worth'] == max_val)
            ]
            
            if not zone_records:
                continue
            
            # 计算该区间后的未来收益
            up_count = 0
            down_count = 0
            future_returns_5d = []
            future_returns_20d = []
            
            for idx, record in zone_records:
                if idx + 5 < len(self.df):
                    ret_5d = (self.df[idx + 5]['net_worth'] - record['net_worth']) / record['net_worth'] * 100
                    future_returns_5d.append(ret_5d)
                    if ret_5d > 0:
                        up_count += 1
                    else:
                        down_count += 1
                
                if idx + 20 < len(self.df):
                    ret_20d = (self.df[idx + 20]['net_worth'] - record['net_worth']) / record['net_worth'] * 100
                    future_returns_20d.append(ret_20d)
            
            total_signals = len(future_returns_5d)
            if total_signals > 0:
                up_prob = up_count / total_signals * 100
                down_prob = down_count / total_signals * 100
                
                # 判断区间类型
                if up_prob >= 60:
                    zone_type = 'support'
                    zone_type_cn = '🟢支撑'
                elif down_prob >= 60:
                    zone_type = 'resistance'
                    zone_type_cn = '🔴压力'
                else:
                    zone_type = 'neutral'
                    zone_type_cn = '⚪中性'
                
                zones.append({
                    'zone_index': i,
                    'price_range': f"{zone_min:.4f} - {zone_max:.4f}",
                    'mid_price': zone_mid,
                    'up_probability': round(up_prob, 1),
                    'down_probability': round(down_prob, 1),
                    'avg_return_5d': round(float(np.mean(future_returns_5d)), 2) if future_returns_5d else 0,
                    'avg_return_20d': round(float(np.mean(future_returns_20d)), 2) if future_returns_20d else 0,
                    'zone_type': zone_type,
                    'zone_type_cn': zone_type_cn
                })
        
        # 当前所在区间
        current_zone_idx = int((current_price - min_val) / zone_size)
        current_zone_idx = min(current_zone_idx, num_zones - 1)
        
        # 获取当前净值附近的区间（前后各2个区间，共5个）
        nearby_zones = []
        for zone in zones:
            if abs(zone['zone_index'] - current_zone_idx) <= 2:
                nearby_zones.append(zone)
        
        return {
            'current_price': current_price,
            'current_position': (current_price - min_val) / (max_val - min_val) * 100,
            'price_range': {'min': min_val, 'max': max_val},
            'current_zone_index': current_zone_idx,
            'nearby_zones': nearby_zones,
            'all_zones': zones
        }

In [ ]:
class FeishuNotifier:
    """飞书通知器"""
    
    def __init__(self, webhook_url: str):
        self.webhook_url = webhook_url
    
    def send_fund_notification(self, fund_results: List[Dict]) -> bool:
        """发送基金监控卡片消息"""
        if not self.webhook_url:
            print("警告: Webhook地址未配置，跳过发送通知")
            return False
        
        # 构建卡片内容
        elements = []
        
        # 添加每个基金的信息
        for result in fund_results:
            if not result or 'error' in result:
                continue
            
            fund_name = result.get('name', '未知基金')
            fund_code = result.get('code', '')
            latest = result.get('latest_info', {})
            pressure = result.get('pressure_analysis', {})
            
            # 日涨跌幅颜色
            daily_return = latest.get('daily_return', 0)
            return_color = "red" if daily_return >= 0 else "green"
            return_text = f"+{daily_return:.2f}%" if daily_return >= 0 else f"{daily_return:.2f}%"
            
            # 净值日期
            net_worth_date = latest.get('date', '未知')
            
            # 位置百分比
            position = pressure.get('current_position', 0)
            position_bar = self._generate_position_bar(position)
            
            # 当前净值附近的区间信息
            nearby_zones = pressure.get('nearby_zones', [])
            zones_text = self._format_nearby_zones(nearby_zones, pressure.get('current_zone_index', 0))
            
            elements.extend([
                {
                    "tag": "div",
                    "text": {
                        "tag": "lark_md",
                        "content": f"**{fund_name}** ({fund_code})"
                    }
                },
                {
                    "tag": "div",
                    "text": {
                        "tag": "lark_md",
                        "content": f"💰 最新净值({net_worth_date}): **{latest.get('net_worth', 0):.4f}** | 日涨跌: <font color='{return_color}'>{return_text}</font>"
                    }
                },
                {
                    "tag": "div",
                    "text": {
                        "tag": "lark_md",
                        "content": f"📊 历史位置: {position:.1f}% {position_bar}"
                    }
                },
                {
                    "tag": "div",
                    "text": {
                        "tag": "lark_md",
                        "content": f"📈 附近区间(当前→):\n{zones_text}"
                    }
                },
                {"tag": "hr"}
            ])
        
        if not elements:
            print("没有有效的基金数据可发送")
            return False
        
        # 移除最后一个分割线
        elements = elements[:-1]
        
        # 构建卡片消息
        card_data = {
            "msg_type": "interactive",
            "card": {
                "config": {
                    "wide_screen_mode": True
                },
                "header": {
                    "title": {
                        "tag": "plain_text",
                        "content": f"📈 基金每日监控报告 - {datetime.now().strftime('%Y-%m-%d')}"
                    },
                    "template": "blue"
                },
                "elements": elements
            }
        }
        
        try:
            response = requests.post(
                self.webhook_url,
                json=card_data,
                headers={'Content-Type': 'application/json'},
                timeout=30
            )
            
            if response.status_code == 200:
                result = response.json()
                if result.get('code') == 0:
                    print("✅ 飞书通知发送成功")
                    return True
                else:
                    print(f"❌ 飞书通知发送失败: {result}")
                    return False
            else:
                print(f"❌ 飞书通知发送失败: HTTP {response.status_code}")
                return False
        except Exception as e:
            print(f"❌ 飞书通知发送异常: {str(e)}")
            return False
    
    def _generate_position_bar(self, percentage: float) -> str:
        """生成位置进度条"""
        filled = int(percentage / 10)
        empty = 10 - filled
        return "█" * filled + "░" * empty
    
    def _format_nearby_zones(self, zones: List[Dict], current_idx: int) -> str:
        """格式化附近区间信息"""
        if not zones:
            return "无数据"
        
        lines = []
        for zone in sorted(zones, key=lambda x: x['zone_index']):
            is_current = zone['zone_index'] == current_idx
            prefix = "👉" if is_current else "  "
            zone_info = f"{prefix} {zone['price_range']} {zone['zone_type_cn']}"
            lines.append(zone_info)
        return "\n".join(lines)

In [ ]:
def analyze_funds(fund_codes: List[str]) -> List[Dict]:
    """分析多个基金"""
    results = []
    
    for code in fund_codes:
        print(f"\n正在分析基金: {code}")
        
        # 获取数据
        fetcher = FundDataFetcher(code)
        data = fetcher.fetch()
        
        if not data:
            print(f"  ❌ 获取数据失败")
            results.append({'code': code, 'error': '获取数据失败'})
            continue
        
        print(f"  ✅ 获取数据成功: {data.get('name', '')}")
        
        # 分析数据
        analyzer = FundAnalyzer(data)
        latest_info = analyzer.get_latest_info()
        pressure_analysis = analyzer.analyze_pressure_zones()
        
        # 整合结果
        result = {
            'code': code,
            'name': data.get('name', ''),
            'latest_info': latest_info,
            'pressure_analysis': pressure_analysis,
            'returns': {
                '1m': data.get('return_1m', 0),
                '3m': data.get('return_3m', 0),
                '6m': data.get('return_6m', 0),
                '1y': data.get('return_1y', 0)
            }
        }
        
        # 打印摘要
        print(f"  📅 最新净值日期: {latest_info.get('date')}")
        print(f"  💰 最新净值: {latest_info.get('net_worth', 0):.4f}")
        print(f"  📈 日涨跌幅: {latest_info.get('daily_return', 0):.2f}%")
        print(f"  📊 历史位置: {pressure_analysis.get('current_position', 0):.1f}%")
        
        # 打印附近区间
        nearby_zones = pressure_analysis.get('nearby_zones', [])
        if nearby_zones:
            print(f"  📈 附近区间:")
            current_idx = pressure_analysis.get('current_zone_index', 0)
            for zone in sorted(nearby_zones, key=lambda x: x['zone_index']):
                marker = "👉" if zone['zone_index'] == current_idx else "  "
                print(f"    {marker} {zone['price_range']} {zone['zone_type_cn']}")
        
        results.append(result)
        
        # 添加延迟，避免请求过快
        time.sleep(1)
    
    return results

In [ ]:
# ============================================
# 主执行逻辑
# ============================================

print("=" * 50)
print(f"📈 基金每日监控 - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 50)

# 检查配置
if not FUND_CODES:
    print("⚠️ 警告: 未配置任何基金代码，请在FUND_CODES列表中添加基金代码")
else:
    print(f"📋 监控基金数量: {len(FUND_CODES)}")
    print(f"📋 基金代码: {', '.join(FUND_CODES)}")
    
    # 执行分析
    results = analyze_funds(FUND_CODES)
    
    # 发送通知
    if ENABLE_NOTIFICATION and FEISHU_WEBHOOK_URL:
        print("\n" + "=" * 50)
        print("📤 正在发送飞书通知...")
        notifier = FeishuNotifier(FEISHU_WEBHOOK_URL)
        notifier.send_fund_notification(results)
    elif not FEISHU_WEBHOOK_URL:
        print("\n⚠️ 未配置飞书Webhook地址，跳过发送通知")
        print("   请在配置区域填写FEISHU_WEBHOOK_URL")
    else:
        print("\nℹ️ 通知已禁用，仅展示数据")

print("\n" + "=" * 50)
print("✅ 执行完成")
print("=" * 50)

## Kaggle定时调度配置

1. 保存此Notebook
2. 在右侧边栏点击 "Save Version"
3. 选择 "Save & Run All"
4. 等待执行完成后，点击 "Publish"
5. 进入Kaggle个人主页 → 找到此Notebook
6. 点击 "..." → "Schedule"
7. 设置定时规则：
   - Frequency: Daily（每日）
   - Time: 20:00（晚上8点，建议交易日收盘后执行）
   - Timezone: Asia/Shanghai（北京时间）
8. 点击 "Create Schedule"

## 注意事项

1. **Webhook地址安全**: 建议在Kaggle的Secrets中存储Webhook地址，避免泄露
2. **执行时间**: 建议在交易日晚上8点后执行，确保净值数据已更新
3. **基金代码**: 请输入6位数字基金代码
4. **通知频率**: 每个交易日都会收到通知，节假日无数据时会跳过